In [10]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName('capstone 01').getOrCreate()

In [11]:
from google.colab import files

upload_files = files.upload()

Saving sales.csv to sales.csv
Saving suppliers.json to suppliers.json
Saving inventory.csv to inventory.csv
Saving products.csv to products.csv
Saving stores.csv to stores.csv


In [12]:
from pyspark.sql.functions import col, max, min, sum, avg, when

stores_df = spark.read.csv(
    'stores.csv',
    header = True,
    inferSchema = True
)

products_df = spark.read.csv(
    'products.csv',
    header = True,
    inferSchema = True
)

inventory_df = spark.read.csv(
    'inventory.csv',
    header = True,
    inferSchema = True
)

sales_df = spark.read.csv(
    'sales.csv',
    header = True,
    inferSchema = True
)

suppliers_df = spark.read.option(
    'multiline',
    'true'
).json('suppliers.json')

In [13]:
stores_df.printSchema()
products_df.printSchema()
inventory_df.printSchema()
sales_df.printSchema()
suppliers_df.printSchema()

print('stores rec count : ', stores_df.count())
print('products rec count : ', products_df.count())
print('inventory rec count : ', inventory_df.count())
print('sales rec count : ', sales_df.count())
print('suppliers rec count : ', suppliers_df.count())

root
 |-- store_id: string (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- store_type: string (nullable = true)
 |-- manager_name: string (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- supplier_id: string (nullable = true)
 |-- unit_price: integer (nullable = true)

root
 |-- inventory_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- last_update: date (nullable = true)

root
 |-- sale_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity_sold: integer (nullable = true)
 |-- sale_amount: int

In [14]:
!mkdir -p bronze

stores_df.write.mode('overwrite').parquet('bronze/stores')
products_df.write.mode('overwrite').parquet('bronze/products')
inventory_df.write.mode('overwrite').parquet('bronze/inventory')
sales_df.write.mode('overwrite').parquet('bronze/sales')
suppliers_df.write.mode('overwrite').parquet('bronze/suppliers')

In [16]:
products_df.filter(col('supplier_id').isNull()).show()
inventory_df.filter(col('stock_quantity').isNull()).show()
sales_df.filter(col('sale_amount').isNull()).show()
sales_df.filter(col('payment_mode').isNull()).show()

inventory_df = inventory_df.na.fill({
    'stock_quantity': 0
})
sales_df = sales_df.na.fill({
    'sale_amount': 0,
    'payment_mode': 'Not Provided'
})
products_df = products_df.na.fill({
    'supplier_id': 'UNKNOWN'
})

+----------+------------+--------+-----+-----------+----------+
|product_id|product_name|category|brand|supplier_id|unit_price|
+----------+------------+--------+-----+-----------+----------+
|      P112|     T-Shirt| Fashion| Puma|       NULL|      1500|
+----------+------------+--------+-----+-----------+----------+

+------------+--------+----------+--------------+-------------+-----------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|
+------------+--------+----------+--------------+-------------+-----------+
|       I1010|    S106|      P109|          NULL|            6| 2026-01-16|
+------------+--------+----------+--------------+-------------+-----------+

+-------+--------+----------+----------+-------------+-----------+------------+
|sale_id|store_id|product_id| sale_date|quantity_sold|sale_amount|payment_mode|
+-------+--------+----------+----------+-------------+-----------+------------+
| SA1011|    S102|      P103|2026-01-17|            1|    

In [18]:
from functools import reduce

def itr_lst(df):
  lst = []
  for c in df.columns:
    lst.append(col(c).isNull())
  return lst

prod_condition = reduce(lambda a,b : a|b, itr_lst(products_df))
inven_condition = reduce(lambda a,b : a|b, itr_lst(inventory_df))
sale_condition = reduce(lambda a,b : a|b, itr_lst(sales_df))

products_df = products_df.withColumn(
    'data_quality_status',
    when(prod_condition, 'Incomplete')
    .otherwise('Complete')
)

inventory_df = inventory_df.withColumn(
    'data_quality_status',
    when(inven_condition, 'Incomplete')
    .otherwise('Complete')
)

sales_df = sales_df.withColumn(
    'data_quality_status',
    when(sale_condition, 'Incomplete')
    .otherwise('Complete')
)

In [19]:
!mkdir -p silver

stores_df.write.mode('overwrite').parquet('silver/stores')
products_df.write.mode('overwrite').parquet('silver/products')
inventory_df.write.mode('overwrite').parquet('silver/inventory')
sales_df.write.mode('overwrite').parquet('silver/sales')
suppliers_df.write.mode('overwrite').parquet('silver/suppliers')

In [20]:
suppliers_df.show()

+---------+--------------------+------+-----------+--------------------+
|     city|             contact|rating|supplier_id|       supplier_name|
+---------+--------------------+------+-----------+--------------------+
|Hyderabad|{techsource@mail....|   4.5|       S201|    TechSource India|
|Bangalore|{mobileworld@mail...|   4.2|       S202|MobileWorld Distr...|
|   Mumbai|  {NULL, 9876500013}|   4.4|       S203|     HomeTech Supply|
|    Delhi|{urban@mail.com, ...|   4.0|       S204|  Urban Furniture Co|
|     Pune|        {NULL, NULL}|   3.8|       S205|      Fashion Direct|
+---------+--------------------+------+-----------+--------------------+



In [21]:
flat_suppliers_df = suppliers_df.select(
    'city',
    col('contact.email').alias('email'),
    col('contact.phone').alias('phone'),
    'rating',
    'supplier_id',
    'supplier_name'
)
flat_suppliers_df.show()

+---------+--------------------+----------+------+-----------+--------------------+
|     city|               email|     phone|rating|supplier_id|       supplier_name|
+---------+--------------------+----------+------+-----------+--------------------+
|Hyderabad| techsource@mail.com|9876500011|   4.5|       S201|    TechSource India|
|Bangalore|mobileworld@mail.com|      NULL|   4.2|       S202|MobileWorld Distr...|
|   Mumbai|                NULL|9876500013|   4.4|       S203|     HomeTech Supply|
|    Delhi|      urban@mail.com|9876500014|   4.0|       S204|  Urban Furniture Co|
|     Pune|                NULL|      NULL|   3.8|       S205|      Fashion Direct|
+---------+--------------------+----------+------+-----------+--------------------+



In [23]:
flat_suppliers_df.select('phone').show()
flat_suppliers_df.select('email').show()
flat_suppliers_df = flat_suppliers_df.na.fill({
    'phone': 'Not Provided',
    'email': 'Not Provided'
})

flat_suppliers_df.write.mode('overwrite').parquet('flat_suppliers_df.parquet')
flat_suppliers_df.show()

+------------+
|       phone|
+------------+
|  9876500011|
|Not Provided|
|  9876500013|
|  9876500014|
|Not Provided|
+------------+

+--------------------+
|               email|
+--------------------+
| techsource@mail.com|
|mobileworld@mail.com|
|        Not Provided|
|      urban@mail.com|
|        Not Provided|
+--------------------+

+---------+--------------------+------------+------+-----------+--------------------+
|     city|               email|       phone|rating|supplier_id|       supplier_name|
+---------+--------------------+------------+------+-----------+--------------------+
|Hyderabad| techsource@mail.com|  9876500011|   4.5|       S201|    TechSource India|
|Bangalore|mobileworld@mail.com|Not Provided|   4.2|       S202|MobileWorld Distr...|
|   Mumbai|        Not Provided|  9876500013|   4.4|       S203|     HomeTech Supply|
|    Delhi|      urban@mail.com|  9876500014|   4.0|       S204|  Urban Furniture Co|
|     Pune|        Not Provided|Not Provided|   3.8|  

In [25]:
products_df.join(
    suppliers_df,
    'supplier_id',
    'left'
).show()

products_df.join(
    inventory_df,
    'product_id',
    'right'
).show()

sales_df.join(
    stores_df,
    'store_id',
    'left'
).show()

sales_df.join(
    products_df,
    'product_id',
    'left'
).show()

full_sales = sales_df.join(
    products_df,
    'product_id',
    'left'
).join(
    stores_df,
    'store_id',
    'left'
)

des_suppliers_df = suppliers_df.select('supplier_id')
products_df.join(
    des_suppliers_df,
    'supplier_id',
    'left_anti'
).show()

des_products_df = products_df.select('product_id')
inventory_df.join(
    des_products_df,
    'product_id',
    'left_anti'
).show()

sales_df.join(
    des_products_df,
    'product_id',
    'left_anti'
).show()

des_stores_df = stores_df.select('store_id')
sales_df.join(
    des_stores_df,
    'store_id',
    'left_anti'
).show()

+-----------+----------+------------+-----------+------------+----------+-------------------+---------+--------------------+------+--------------------+
|supplier_id|product_id|product_name|   category|       brand|unit_price|data_quality_status|     city|             contact|rating|       supplier_name|
+-----------+----------+------------+-----------+------------+----------+-------------------+---------+--------------------+------+--------------------+
|       S201|      P101|      Laptop|Electronics|      Lenovo|     65000|           Complete|Hyderabad|{techsource@mail....|   4.5|    TechSource India|
|       S202|      P102|      Mobile|Electronics|     Samsung|     25000|           Complete|Bangalore|{mobileworld@mail...|   4.2|MobileWorld Distr...|
|       S203|      P103|  Television|Electronics|          LG|     45000|           Complete|   Mumbai|  {NULL, 9876500013}|   4.4|     HomeTech Supply|
|       S204|      P104|Office Chair|  Furniture| Featherlite|      7000|         

In [27]:
from pyspark.sql.functions import *

inventory_df.withColumn(
    'stock_status',
    when(col('stock_quantity') <= col('reorder_level'), 'Reorder Required')
    .otherwise('Sufficient Stock')
).show()

products_df.withColumn(
    'price_category',
    when(col('unit_price') >= 50000, 'Premium')
    .when(col('unit_price') >= 10000, 'Standard')
    .otherwise('Budget')
).show()

sales_df.withColumn(
    'revenue_category',
    when(col('sale_amount') >= 50000, 'High Revenue')
    .when(col('sale_amount') >= 15000, 'Medium Revenue')
    .otherwise('Low Revenue')
).show()

dit = {1:'jan', 2:'feb', 3:'mar', 4:'apr', 5:'may', 6:'jun', 7:'jul', 8:'aug', 9:'sep', 10:'oct', 11:'nov', 12:'dec'}
sales_df.withColumn(
    'month',
    date_format('sale_date', 'MMM')
).show()

sales_df.withColumn(
    'year',
    year('sale_date')
).show()

inventory_df.join(
    products_df,
    'product_id',
    'left'
).withColumn(
    'inventory_value',
    col('stock_quantity') * col('unit_price')
).show()

suppliers_df.withColumn(
    'supplier_quality',
    when(col('rating')>=4.5, 'Excellent')
    .when(col('rating')>=4.0, 'Good')
    .otherwise('Average')
).show()

+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+
|inventory_id|store_id|product_id|stock_quantity|reorder_level|last_update|data_quality_status|    stock_status|
+------------+--------+----------+--------------+-------------+-----------+-------------------+----------------+
|       I1001|    S101|      P101|            10|            5| 2026-01-10|           Complete|Sufficient Stock|
|       I1002|    S101|      P102|            25|           10| 2026-01-10|           Complete|Sufficient Stock|
|       I1003|    S101|      P104|             3|            5| 2026-01-11|           Complete|Reorder Required|
|       I1004|    S102|      P101|             8|            5| 2026-01-12|           Complete|Sufficient Stock|
|       I1005|    S102|      P103|             5|            4| 2026-01-12|           Complete|Sufficient Stock|
|       I1006|    S103|      P105|             2|            5| 2026-01-13|           Complete|R

In [30]:
stores_df.groupBy('city').agg(count('*').alias('store count')).show()
products_df.groupBy('category').agg(count('product_id').alias('product count')).show()
products_df.groupBy('brand').agg(count('product_id').alias('product count')).show()

inventory_df.join(
    products_df.select('product_id', 'unit_price'),
    'product_id',
    'left'
).groupBy('store_id').agg(
    sum(
        col('stock_quantity') * col('unit_price')
    ).alias('inventory_value')
).show()

inventory_df.join(
    products_df.select('product_id', 'unit_price', 'category'),
    'product_id',
    'left'
).groupBy('category').agg(
    sum(
        col('stock_quantity') * col('unit_price')
    ).alias('inventory_value')
).show()

inventory_df.filter(
    col('stock_quantity') <= col('reorder_level')
).groupBy('product_id').agg(
    count('*').alias('reorder_needed')
).show()

sales_df.agg(sum('sale_amount').alias('total revenue')).show()
sales_df.groupBy('store_id').agg(sum('sale_amount').alias('revenue')).show()
sales_df.join(
    stores_df.select('store_id', 'city'),
    'store_id',
    'left'
).groupBy('city').agg(sum('sale_amount').alias('revenue')).show()
sales_df.join(
    products_df.select('product_id', 'category'),
    'product_id',
    'left'
).groupBy('category').agg(sum('sale_amount').alias('revenue')).show()
sales_df.groupBy('product_id').agg(sum('sale_amount').alias('revenue')).show()
sales_df.groupBy('payment_mode').agg(sum('sale_amount').alias('revenue')).show()
hig_rev_prod = sales_df.groupBy('product_id').agg(sum('sale_amount').alias('revenue')).orderBy(col('revenue').desc()).select('product_id').limit(1)
products_df.join(
    hig_rev_prod,
    'product_id',
    'inner'
).show()
hig_rev_sto = sales_df.groupBy('store_id').agg(sum('sale_amount').alias('revenue')).orderBy(col('revenue').desc()).select('store_id').limit(1)
stores_df.join(
    hig_rev_sto,
    'store_id',
    'inner'
).show()
print('high revenue categroy : ')
sales_df.join(
    products_df.select('product_id', 'category'),
    'product_id',
    'left'
).groupBy('category').agg(sum('sale_amount').alias('revenue')).orderBy(col('revenue').desc()).select('category').show(1)

+---------+-----------+
|     city|store count|
+---------+-----------+
|Bangalore|          1|
|    Kochi|          1|
|  Chennai|          1|
|   Mumbai|          1|
|     Pune|          1|
|    Delhi|          1|
|Hyderabad|          1|
|   Jaipur|          1|
+---------+-----------+

+-----------+-------------+
|   category|product count|
+-----------+-------------+
|    Fashion|            4|
|Electronics|            5|
|  Furniture|            3|
+-----------+-------------+

+------------+-------------+
|       brand|product count|
+------------+-------------+
|        Nike|            1|
|        Sony|            1|
|Urban Ladder|            1|
|        Puma|            1|
|      Lenovo|            1|
| Featherlite|            1|
|     Samsung|            1|
|      Godrej|            1|
|          LG|            1|
|   Wildcraft|            1|
|    Fastrack|            1|
|   Whirlpool|            1|
+------------+-------------+

+--------+---------------+
|store_id|inventory_va

In [31]:
from pyspark.sql.window import Window as win

win_spec = win.orderBy(col('revenue').desc())

sales_df.join(
    products_df,
    'product_id',
    'left'
).groupby('product_id').agg(sum('sale_amount').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).show()

win_spec = win.orderBy(col('revenue').desc())
sales_df.join(
    stores_df,
    'store_id',
    'left'
).groupBy('store_id').agg(sum('sale_amount').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).show()

win_spec = win.partitionBy('category').orderBy(col('revenue').desc())
sales_df.join(
    products_df,
    'product_id',
    'left'
).groupBy('category', 'product_id').agg(
    sum(col('sale_amount')).alias('revenue')).withColumn(
  'rank',
  rank().over(win_spec)
).show()

win_spec = win.partitionBy('category').orderBy(col('revenue').desc())
sales_df.join(
    products_df,
    'product_id',
    'left'
).groupBy('category', 'product_id').agg(sum('sale_amount').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).filter(col('rank')==1).show()

win_spec = win.partitionBy('category').orderBy(col('revenue').desc())
sales_df.join(
    products_df,
    'product_id',
    'left'
).groupBy('category', 'product_id').agg(sum('sale_amount').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).filter(col('rank')<=3).show()

win_spec = win.orderBy(col('revenue').desc())
sales_df.join(
    stores_df,
    'store_id',
    'left'
).groupBy('city').agg(sum('sale_amount').alias('revenue')).withColumn(
    'rank',
    rank().over(win_spec)
).filter(col('rank')==1).show()

win_spec = win.orderBy('sale_date')
sales_df.withColumn(
    'running_sum',
    sum('sale_amount').over(win_spec)
).show()

sales_df.withColumns({
    'prev': lag('sale_amount').over(win_spec),
    'nex': lead('sale_amount').over(win_spec)
}).select('sale_date', 'sale_amount', 'prev', 'nex').show()

+----------+-------+----+
|product_id|revenue|rank|
+----------+-------+----+
|      P101| 130000|   1|
|      P102|  75000|   2|
|      P109|  38000|   3|
|      P110|  32000|   4|
|      P107|  24000|   5|
|      P108|  20000|   6|
|      P106|  18000|   7|
|      P104|  14000|   8|
|      P105|  12000|   9|
|      P120|  10000|  10|
|      P103|      0|  11|
+----------+-------+----+

+--------+-------+----+
|store_id|revenue|rank|
+--------+-------+----+
|    S101| 154000|   1|
|    S102|  65000|   2|
|    S106|  38000|   3|
|    S107|  32000|   4|
|    S103|  30000|   5|
|    S104|  24000|   6|
|    S105|  20000|   7|
|    S108|  10000|   8|
+--------+-------+----+

+-----------+----------+-------+----+
|   category|product_id|revenue|rank|
+-----------+----------+-------+----+
|       NULL|      P120|  10000|   1|
|Electronics|      P101| 130000|   1|
|Electronics|      P102|  75000|   2|
|Electronics|      P109|  38000|   3|
|Electronics|      P103|      0|   4|
|    Fashion|   

In [35]:
win_spec = win.partitionBy('product_id').orderBy('sale_date')
filt = sales_df.withColumn(
    'prev',
    lag('sale_amount').over(win_spec)
).filter(col('sale_amount') > col('prev')).select('product_id').distinct()

products_df.join(
    filt,
    'product_id',
    'inner'
).show()

+----------+------------+--------+--------+-----------+----------+-------------------+
|product_id|product_name|category|   brand|supplier_id|unit_price|data_quality_status|
+----------+------------+--------+--------+-----------+----------+-------------------+
|      P107|       Watch| Fashion|Fastrack|       S206|      8000|           Complete|
+----------+------------+--------+--------+-----------+----------+-------------------+



In [36]:
stores_df.createOrReplaceTempView('stores')
products_df.createOrReplaceTempView('products')
inventory_df.createOrReplaceTempView('inventory')
sales_df.createOrReplaceTempView('sales')
suppliers_df.createOrReplaceTempView('suppliers')

In [41]:
spark.sql("""
  select category, count(*) as products_count from products
  group by category
""").show()
spark.sql("""
  select store_id, sum(sale_amount) as revenue from sales
  group by store_id
""").show()
spark.sql("""
  select city, sum(sale_amount) as revenue from sales sa
  join stores st on st.store_id =sa.store_id
  group by city
""").show()
spark.sql("""
  select * from products
  where product_id in (
    select product_id from inventory
    where stock_quantity <= reorder_level
    )
""").show()
spark.sql("""
  select * from sales
  where product_id not in (
    select product_id from products
    )
""").show()
spark.sql("""
  select * from products
  where supplier_id not in (
    select supplier_id from suppliers
  ) or supplier_id is null
""").show()
spark.sql("""
  select product_id,
        sum(sale_amount) as revenue
  from sales
  group by product_id
  order by revenue desc
  limit 5
""").show()
spark.sql("""
  select payment_mode, sum(sale_amount) as revenue from sales
  group by payment_mode
""").show()

+-----------+--------------+
|   category|products_count|
+-----------+--------------+
|    Fashion|             4|
|Electronics|             5|
|  Furniture|             3|
+-----------+--------------+

+--------+-------+
|store_id|revenue|
+--------+-------+
|    S105|  20000|
|    S102|  65000|
|    S106|  38000|
|    S104|  24000|
|    S107|  32000|
|    S101| 154000|
|    S108|  10000|
|    S103|  30000|
+--------+-------+

+---------+-------+
|     city|revenue|
+---------+-------+
|Bangalore|  65000|
|    Kochi|  32000|
|  Chennai|  24000|
|   Mumbai|  30000|
|     Pune|  38000|
|    Delhi|  20000|
|Hyderabad| 154000|
|   Jaipur|  10000|
+---------+-------+

+----------+------------+-----------+------------+-----------+----------+-------------------+
|product_id|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|
+----------+------------+-----------+------------+-----------+----------+-------------------+
|      P104|Office Chair|  Furniture| Feathe

In [46]:
full_sales.show()
full_sales = full_sales.drop('data_quality_status')

+--------+----------+-------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+--------------------+---------+-----------+-----------+------------+
|store_id|product_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|data_quality_status|product_name|   category|       brand|supplier_id|unit_price|data_quality_status|          store_name|     city|      state| store_type|manager_name|
+--------+----------+-------+----------+-------------+-----------+------------+-------------------+------------+-----------+------------+-----------+----------+-------------------+--------------------+---------+-----------+-----------+------------+
|    S101|      P101| SA1001|2026-01-10|            1|      65000|         UPI|           Complete|      Laptop|Electronics|      Lenovo|       S201|     65000|           Complete|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|
|   

In [47]:
!mkdir -p gold
full_sales.write.mode('overwrite').parquet('gold/full_sales')

In [48]:
full_sales = full_sales.withColumns({
    'month': month('sale_date'),
    'year': year('sale_date')
})
full_sales.write.mode('overwrite').partitionBy('year', 'month').parquet('gold/final_stage')

In [49]:
full_sales.withColumn(
    'day',
    day('sale_date')
).write.mode('overwrite').partitionBy('day').parquet('gold/incremental_day')

In [50]:
read_sales = spark.read.parquet('gold/incremental_day')
read_sales.show()

+--------+----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+--------------------+---------+-----------+-----------+------------+-----+----+---+
|store_id|product_id|sale_id| sale_date|quantity_sold|sale_amount|payment_mode|product_name|   category|       brand|supplier_id|unit_price|          store_name|     city|      state| store_type|manager_name|month|year|day|
+--------+----------+-------+----------+-------------+-----------+------------+------------+-----------+------------+-----------+----------+--------------------+---------+-----------+-----------+------------+-----+----+---+
|    S101|      P101| SA1001|2026-01-10|            1|      65000|         UPI|      Laptop|Electronics|      Lenovo|       S201|     65000|Metro Mart Hyderabad|Hyderabad|  Telangana|Supermarket|Rahul Sharma|    1|2026| 10|
|    S101|      P102| SA1002|2026-01-10|            2|      50000|        Card|      Mobile|Electronics|

In [51]:
before_count = spark.read.parquet("silver/sales").count()

print("Before Incremental Load:", before_count)

Before Incremental Load: 15


In [52]:
read_sales.write.mode('append').parquet('silver/sales')

In [53]:
recal_prod_rev = sales_df.join(
    products_df,
    'product_id',
    'left'
).groupBy('product_id').agg(sum('sale_amount').alias('revenue'))

recal_store_rev = sales_df.join(
    stores_df,
    'store_id',
    'left'
).groupBy('store_id').agg(sum('sale_amount').alias('revenue'))

recal_prod_rev.write.mode('overwrite').parquet('gold/recal_prod_rev')
recal_store_rev.write.mode('overwrite').parquet('gold/recal_store_rev')

In [54]:
full_sales.write.mode('overwrite').partitionBy('year', 'month').parquet('gold/final_stage')

In [55]:
after_count = spark.read.parquet("silver/sales").count()

print("After Incremental Load:", after_count)

After Incremental Load: 30


In [56]:
print("Before Count :", before_count)
print("After Count  :", after_count)
print("Rows Added   :", after_count - before_count)

Before Count : 15
After Count  : 30
Rows Added   : 15


In [59]:
stores_df.join(
    sales_df,
    'store_id',
    'left'
).groupBy(
    'store_id',
    'store_name',
    'city',
    'state'
    ).agg(
      count('*').alias('total_sales'),
      sum('sale_amount').alias('revenue')
    ).select('store_id', 'store_name', 'city', 'state', 'total_sales', 'revenue').show()

+--------+--------------------+---------+-----------+-----------+-------+
|store_id|          store_name|     city|      state|total_sales|revenue|
+--------+--------------------+---------+-----------+-----------+-------+
|    S106|     Metro Mart Pune|     Pune|Maharashtra|          1|  38000|
|    S107|    Metro Mart Kochi|    Kochi|     Kerala|          1|  32000|
|    S101|Metro Mart Hyderabad|Hyderabad|  Telangana|          4| 154000|
|    S103|   Metro Mart Mumbai|   Mumbai|Maharashtra|          2|  30000|
|    S105|    Metro Mart Delhi|    Delhi|      Delhi|          2|  20000|
|    S102|Metro Mart Bangalore|Bangalore|  Karnataka|          2|  65000|
|    S104|  Metro Mart Chennai|  Chennai| Tamil Nadu|          2|  24000|
|    S108|   Metro Mart Jaipur|   Jaipur|  Rajasthan|          1|  10000|
+--------+--------------------+---------+-----------+-----------+-------+



In [62]:
products_df.join(
    sales_df,
    'product_id',
    'left'
).groupBy(
    'product_id',
    'product_name',
    'brand',
    'category'
    ).agg(
      sum('quantity_sold').alias('total_sales'),
      sum('sale_amount').alias('revenue')
    ).select('product_id', 'product_name', 'brand', 'category', 'total_sales', 'revenue').show()

+----------+------------+------------+-----------+-----------+-------+
|product_id|product_name|       brand|   category|total_sales|revenue|
+----------+------------+------------+-----------+-----------+-------+
|      P106|       Shoes|        Nike|    Fashion|          4|  18000|
|      P111|  Headphones|        Sony|Electronics|       NULL|   NULL|
|      P102|      Mobile|     Samsung|Electronics|          3|  75000|
|      P110|        Sofa|      Godrej|  Furniture|          1|  32000|
|      P107|       Watch|    Fastrack|    Fashion|          3|  24000|
|      P112|     T-Shirt|        Puma|    Fashion|       NULL|   NULL|
|      P108|    Backpack|   Wildcraft|    Fashion|          8|  20000|
|      P103|  Television|          LG|Electronics|          1|      0|
|      P101|      Laptop|      Lenovo|Electronics|          2| 130000|
|      P104|Office Chair| Featherlite|  Furniture|          2|  14000|
|      P105| Study Table|Urban Ladder|  Furniture|          1|  12000|
|     

In [64]:
inventory_df.select(
    'store_id', 'product_id', 'stock_quantity', 'reorder_level',
     when(col('stock_quantity')<=col('reorder_level'), 'reorder needed')
     .otherwise('no reorder needed').alias('stock_status')).show()

+--------+----------+--------------+-------------+-----------------+
|store_id|product_id|stock_quantity|reorder_level|     stock_status|
+--------+----------+--------------+-------------+-----------------+
|    S101|      P101|            10|            5|no reorder needed|
|    S101|      P102|            25|           10|no reorder needed|
|    S101|      P104|             3|            5|   reorder needed|
|    S102|      P101|             8|            5|no reorder needed|
|    S102|      P103|             5|            4|no reorder needed|
|    S103|      P105|             2|            5|   reorder needed|
|    S103|      P106|            30|           10|no reorder needed|
|    S104|      P107|             4|            5|   reorder needed|
|    S105|      P108|            50|           20|no reorder needed|
|    S106|      P109|             0|            6|   reorder needed|
|    S107|      P110|             1|            3|   reorder needed|
|    S108|      P120|            1

In [67]:
suppliers_quantity = products_df.groupby('supplier_id').agg(count('*').alias('supplier_quantity'))
suppliers_df.join(
    suppliers_quantity,
    'supplier_id',
    'left'
).select(
    'supplier_id', 'supplier_name', 'city', 'rating',
    'supplier_quantity', 'contact.phone', 'contact.email'
).show()

+-----------+--------------------+---------+------+-----------------+----------+--------------------+
|supplier_id|       supplier_name|     city|rating|supplier_quantity|     phone|               email|
+-----------+--------------------+---------+------+-----------------+----------+--------------------+
|       S201|    TechSource India|Hyderabad|   4.5|                1|9876500011| techsource@mail.com|
|       S202|MobileWorld Distr...|Bangalore|   4.2|                1|      NULL|mobileworld@mail.com|
|       S203|     HomeTech Supply|   Mumbai|   4.4|                2|9876500013|                NULL|
|       S204|  Urban Furniture Co|    Delhi|   4.0|                3|9876500014|      urban@mail.com|
|       S205|      Fashion Direct|     Pune|   3.8|                1|      NULL|                NULL|
+-----------+--------------------+---------+------+-----------------+----------+--------------------+



In [70]:
sales_df.join(
    products_df,
    'product_id',
    'left'
).join(
    stores_df,
    'store_id',
    'left'
).groupby('category').agg(
    count('*').alias('total_products'),
    sum('quantity_sold').alias('total_quantity_sold'),
    sum('sale_amount').alias('total_revenue')
).show()

+-----------+--------------+-------------------+-------------+
|   category|total_products|total_quantity_sold|total_revenue|
+-----------+--------------+-------------------+-------------+
|    Fashion|             5|                 15|        62000|
|       NULL|             1|                  2|        10000|
|Electronics|             6|                  7|       243000|
|  Furniture|             3|                  4|        58000|
+-----------+--------------+-------------------+-------------+



In [71]:
sales_df.groupby('payment_mode').agg(
    count('*').alias('total_transactions'),
    sum('sale_amount').alias('total_revenue')
).select('payment_mode', 'total_transactions', 'total_revenue').show()

+------------+------------------+-------------+
|payment_mode|total_transactions|total_revenue|
+------------+------------------+-------------+
|        Card|                 5|       133000|
|        Cash|                 3|        35500|
|Not Provided|                 1|        14000|
|         UPI|                 6|       190500|
+------------+------------------+-------------+

